test file a01

# validate_preprocessing.py
import pandas as pd
from constant import CLEAN_AUDIT_LOG_PATH, CLEAN_USER_METADATA_PATH

def run_validation():
    print("=== BẮT ĐẦU KIỂM CHỨNG DỮ LIỆU TIỀN XỬ LÝ ===")
    audit = pd.read_csv(CLEAN_AUDIT_LOG_PATH)
    user = pd.read_csv(CLEAN_USER_METADATA_PATH)

    # 1. Kiểm tra giá trị Trống (Cấm tuyệt đối đối với mô hình GNN/GRU)
    assert audit['time_window_norm'].isna().sum() == 0, "LỖI: time_window_norm chứa giá trị NaN!"
    assert user['behavioral_baseline_hour'].isna().sum() == 0, "LỖI: behavioral_baseline_hour chứa giá trị NaN!"
    print("✔ Pass Bước 1: Không có giá trị thiếu (NaN) trong các đặc trưng cốt lõi.")

    # 2. Kiểm tra logic Giờ hành chính (is_after_hours)
    audit['timestamp'] = pd.to_datetime(audit['timestamp'])
    night_logs = audit[audit['timestamp'].dt.hour == 2] # Lấy log lúc 2 giờ sáng
    if not night_logs.empty:
        assert (night_logs['is_after_hours'] == 1).all(), "LỖI: Log lúc 2h sáng nhưng is_after_hours không bằng 1!"
    
    day_logs = audit[(audit['timestamp'].dt.hour == 10) & (audit['timestamp'].dt.minute == 0)] # Lúc 10h00 sáng
    if not day_logs.empty:
        assert (day_logs['is_after_hours'] == 0).all(), "LỖI: Log lúc 10h sáng nhưng is_after_hours không bằng 0!"
    print("✔ Pass Bước 2: Logic phân tách cờ ngoài giờ (is_after_hours) hoạt động chính xác.")

    # 3. Kiểm tra dải giá trị chuẩn hóa (Normalization Bounds)
    assert audit['time_window_norm'].min() >= 0.0 and audit['time_window_norm'].max() <= 1.0, "LỖI: time_window_norm vượt quá dải [0, 1]!"
    print("✔ Pass Bước 3: Đặc trưng thời gian nút Session được chuẩn hóa chính xác về dải [0, 1].")

    # 4. Kiểm tra thứ tự tăng tiến của sequence_order
    sample_session = audit['session_id'].iloc[0]
    session_group = audit[audit['session_id'] == sample_session]
    orders = session_group['sequence_order'].tolist()
    assert orders == sorted(orders), "LỖI: sequence_order không được sắp xếp tăng dần theo thời gian!"
    print("✔ Pass Bước 4: Cạnh BELONGS_TO có sequence_order tuần tự chuẩn.")

    # 5. In báo cáo tổng quan phân phối
    print("\n=== THỐNG KÊ PHÂN PHỐI ĐẶC TRƯNG ===")
    print(f"- Tỷ lệ hành động ngoài giờ: {audit['is_after_hours'].mean() * 100:.2f}%")
    print(f"- Tỷ lệ truy cập từ IP nội bộ: {audit['is_internal_ip'].mean() * 100:.2f}%")
    print(f"- Phân phối baseline giờ của User:\n{user['behavioral_baseline_hour'].describe()}")
    print("\n[KẾT LUẬN] Toàn bộ dữ liệu tiền xử lý đạt chuẩn cấu trúc cho mô hình GNN + GRU.")

if __name__ == "__main__":
    run_validation()

In [2]:
import pandas as pd
import numpy as np
import datetime
from constant import CLEAN_AUDIT_LOG_PATH, CLEAN_USER_METADATA_PATH

def run_validation():
    print("====================================================")
    print("=== BẮT ĐẦU KIỂM CHỨNG TOÀN DIỆN DỮ LIỆU SẠCH ===")
    print("====================================================")
    
    # Đọc tập dữ liệu sạch sau tiền xử lý
    audit = pd.read_csv(CLEAN_AUDIT_LOG_PATH)
    user = pd.read_csv(CLEAN_USER_METADATA_PATH)
    
    # Chuyển đổi định dạng thời gian chuẩn để kiểm tra
    audit['timestamp'] = pd.to_datetime(audit['timestamp'])
    audit['time_only'] = audit['timestamp'].dt.time

    # --------------------------------------------------------------
    # 1. KIỂM TRA GIÁ TRỊ THIẾU (CRITICAL FOR GRAPH/GRU)
    # --------------------------------------------------------------
    critical_audit_cols = ['session_id', 'db_user', 'cmd_type', 'time_window_norm', 
                           'is_internal_ip', 'is_after_hours', 'sequence_order']
    critical_user_cols = ['db_user', 'role_index', 'department_index', 'behavioral_baseline_hour']
    
    # Kiểm tra NaN trên toàn bộ các cột đặc trưng cốt lõi
    nan_audit = audit[critical_audit_cols].isna().sum().sum()
    nan_user = user[critical_user_cols].isna().sum().sum()
    
    assert nan_audit == 0, f"LỖI: Phát hiện có giá trị NaN trong các thuộc tính quan trọng của Audit Log!"
    assert nan_user == 0, f"LỖI: Phát hiện có giá trị NaN trong thuộc tính của nút User!"
    print("✔ [PASSED] Bước 1: Toàn bộ đặc trưng nút và cạnh không chứa giá trị rỗng (NaN).")

    # --------------------------------------------------------------
    # 2. KIỂM TRA ĐỒNG BỘ ĐỊNH DANH (INTEGRITY CHECK)
    # --------------------------------------------------------------
    # Đảm bảo mọi db_user trong audit_logs đều có hồ sơ thực thể nằm trong file user_metadata
    audit_users = set(audit['db_user'].unique())
    metadata_users = set(user['db_user'].unique())
    
    missing_users = audit_users - metadata_users
    assert len(missing_users) == 0, f"LỖI: Phát hiện các user thực thi lệnh nhưng không tồn tại trong danh sách Metadata: {missing_users}"
    print(f"✔ [PASSED] Bước 2: Rà soát thực thể hoàn tất. Tất cả {len(audit_users)} người dùng đều khớp định danh.")

    # --------------------------------------------------------------
    # 3. KIỂM TRA TOÀN BỘ LOGIC GIỜ HÀNH CHÍNH (is_after_hours)
    # --------------------------------------------------------------
    # Định nghĩa chính xác theo cấu hình trong file a01
    start_work = datetime.time(8, 30, 0)
    end_work = datetime.time(17, 45, 0)
    
    # Quét toàn bộ bảng xem có dòng nào bị gán nhãn sai hay không
    wrong_work_hours = audit[
        (audit['time_only'] >= start_work) & 
        (audit['time_only'] < end_work) & 
        (audit['is_after_hours'] != 0)
    ]
    wrong_after_hours = audit[
        ((audit['time_only'] < start_work) | (audit['time_only'] >= end_work)) & 
        (audit['is_after_hours'] != 1)
    ]
    
    assert len(wrong_work_hours) == 0, f"LỖI: Có {len(wrong_work_hours)} dòng trong giờ làm việc nhưng bị gán nhãn ngoài giờ!"
    assert len(wrong_after_hours) == 0, f"LỖI: Có {len(wrong_after_hours)} dòng ngoài giờ làm việc nhưng bị gán nhãn trong giờ!"
    print("✔ [PASSED] Bước 3: Đã quét 100% dữ liệu. Thuộc tính cạnh 'is_after_hours' chính xác tuyệt đối.")

    # --------------------------------------------------------------
    # 4. KIỂM TRA HỢP LỆ MIỀN GIÁ TRỊ (BOUNDS & CATEGORICAL VALIDATION)
    # --------------------------------------------------------------
    # Kiểm tra dải chuẩn hóa [0, 1]
    assert audit['time_window_norm'].min() >= 0.0 and audit['time_window_norm'].max() <= 1.0, "LỖI: time_window_norm vượt dải [0, 1]!"
    
    # Kiểm tra miền mã hóa loại lệnh SQL (cmd_type từ 0 đến 7)
    assert audit['cmd_type'].isin(range(8)).all(), "LỖI: cmd_type chứa mã lệnh nằm ngoài danh mục [0-7]!"
    
    # Kiểm tra dải IP nội bộ (-1: lỗi/nan, 0: external, 1: internal)
    assert audit['is_internal_ip'].isin([-1, 0, 1]).all(), "LỖI: Trường is_internal_ip chứa giá trị dị biệt không thuộc tập [-1, 0, 1]!"
    
    # Kiểm tra miền giá trị của baseline làm việc (phải nằm trong mốc giờ 0 -> 23.x)
    assert user['behavioral_baseline_hour'].min() >= 0.0 and user['behavioral_baseline_hour'].max() < 24.0, "LỖI: Khung giờ baseline bị tính toán vượt ngưỡng ngày!"
    print("✔ [PASSED] Bước 4: Toàn bộ miền dữ liệu phân loại và dải số thực chuẩn hóa đều hợp lệ.")

    # --------------------------------------------------------------
    # 5. KIỂM TRA TOÀN BỘ CHUỖI THỜI GIAN (sequence_order & duration)
    # --------------------------------------------------------------
    # Quét tất cả các session để kiểm tra tính tuần tiến
    print("-> Đang rà soát tính tuần tự của chuỗi sự kiện trong từng phiên...")
    
    # Kiểm tra xem sequence_order có bắt đầu từ 1 không
    session_min_order = audit.groupby('session_id')['sequence_order'].min()
    assert (session_min_order == 1).all(), "LỖI: Có session trích xuất chuỗi câu lệnh không bắt đầu từ thứ tự 1!"
    
    # Kiểm tra xem sequence_order có bị trùng lặp hoặc nhảy cóc trong cùng 1 session không
    # Bằng cách so sánh hiệu mốc thời gian sau khi sắp xếp
    is_ordered_correctly = True
    for sid, group in audit.groupby('session_id'):
        orders = group['sequence_order'].tolist()
        if orders != list(range(1, len(orders) + 1)):
            print(f"LỖI: Session {sid} bị đứt gãy chuỗi thứ tự: {orders}")
            is_ordered_correctly = False
            break
            
    assert is_ordered_correctly, "LỖI: sequence_order trong một hoặc nhiều session không tuần tiến liên tục!"
    
    # Kiểm tra độ dài phiên (duration không được âm)
    assert (audit['session_duration'] >= 0).all(), "LỖI: Phát hiện có session_duration bị âm (Sai lệch logic timestamp)!"
    print("✔ [PASSED] Bước 5: Chuỗi cấu trúc thời gian của Session đạt chuẩn đầu vào cho mạng hồi quy GRU.")

    # --------------------------------------------------------------
    # 6. IN BÁO CÁO PHÂN PHỐI THỐNG KÊ CHI TIẾT
    # --------------------------------------------------------------
    print("\n====================================================")
    print("===            BÁO CÁO PHÂN PHỐI THỐNG KÊ        ===")
    print("====================================================")
    print(f"- Tổng số bản ghi Audit Log sạch  : {len(audit)}")
    print(f"- Tổng số người dùng hệ thống     : {len(user)}")
    print(f"- Tổng số phiên làm việc (Session): {audit['session_id'].nunique()}")
    print(f"- Tỷ lệ hành vi ngoài giờ hành chính: {audit['is_after_hours'].mean() * 100:.2f}%")
    print(f"- Tỷ lệ truy cập từ mạng nội bộ (IP): {audit[audit['is_internal_ip'] == 1].shape[0] / len(audit) * 100:.2f}%")
    print(f"- Tỷ lệ phiên log bị lỗi IP (-1)    : {audit[audit['is_internal_ip'] == -1].shape[0] / len(audit) * 100:.2f}%")
    print(f"- Số lượng hành vi độc hại ghi nhận : {audit['is_anomaly'].sum()} câu lệnh SQL")
    print("\nPhân bổ đặc trưng 'behavioral_baseline_hour' của User:")
    print(user['behavioral_baseline_hour'].describe())
    print("\n[KẾT LUẬN] Tập dữ liệu tiền xử lý ĐẠT CHUẨN AN TOÀN và logic để chuyển sang Bước 2 Dựng Đồ Thị.")
    print("====================================================")

if __name__ == "__main__":
    run_validation()

=== BẮT ĐẦU KIỂM CHỨNG TOÀN DIỆN DỮ LIỆU SẠCH ===
✔ [PASSED] Bước 1: Toàn bộ đặc trưng nút và cạnh không chứa giá trị rỗng (NaN).
✔ [PASSED] Bước 2: Rà soát thực thể hoàn tất. Tất cả 30 người dùng đều khớp định danh.
✔ [PASSED] Bước 3: Đã quét 100% dữ liệu. Thuộc tính cạnh 'is_after_hours' chính xác tuyệt đối.
✔ [PASSED] Bước 4: Toàn bộ miền dữ liệu phân loại và dải số thực chuẩn hóa đều hợp lệ.
-> Đang rà soát tính tuần tự của chuỗi sự kiện trong từng phiên...
✔ [PASSED] Bước 5: Chuỗi cấu trúc thời gian của Session đạt chuẩn đầu vào cho mạng hồi quy GRU.

===            BÁO CÁO PHÂN PHỐI THỐNG KÊ        ===
- Tổng số bản ghi Audit Log sạch  : 1307
- Tổng số người dùng hệ thống     : 30
- Tổng số phiên làm việc (Session): 93
- Tỷ lệ hành vi ngoài giờ hành chính: 100.00%
- Tỷ lệ truy cập từ mạng nội bộ (IP): 100.00%
- Tỷ lệ phiên log bị lỗi IP (-1)    : 0.00%
- Số lượng hành vi độc hại ghi nhận : 285 câu lệnh SQL

Phân bổ đặc trưng 'behavioral_baseline_hour' của User:
count    30.000000

In [5]:
"""
File: verify_graph.py
Nhiệm vụ:
- Chạy hàm build_heterogeneous_graph() từ file a02
- Kiểm tra số lượng chiều (dimension) của từng loại Nút (Node)
- Kiểm tra tính khớp nối số lượng hàng/cột của các thuộc tính Cạnh (Edge Attributes)
- Xác thực ma trận nhãn (Labels) và dữ liệu phân vùng mặt nạ (Masks)
"""

import torch
import sys
import os

# Đảm bảo hệ thống tìm thấy các file trong thư mục hiện hành
# sys.path.append(os.path.dirname(os.path.abspath(__file__)))

try:
    from a02_graph_construction import build_heterogeneous_graph
except ImportError:
    print("[LỖI] Không tìm thấy file a02_graph_construction.py hoặc hàm build_heterogeneous_graph!")
    sys.exit(1)

def run_graph_verification():
    print("====================================================")
    print("▶ BẮT ĐẦU KIỂM TRA ĐỒ THỊ HETERODATA (a02 VS a01)...")
    print("====================================================\n")

    # 1. Khởi chạy build đồ thị
    try:
        data = build_heterogeneous_graph()
        print("\n[OK] Khởi tạo cấu trúc HeteroData thành công!")
    except Exception as e:
        print(f"[THẤT BẠI LỚN] Hàm build_heterogeneous_graph gặp lỗi khi chạy: {e}")
        return

    errors = 0

    # ================================================================
    # KHẢO SÁT NÚT (NODE FEATURES CHECK)
    # ================================================================
    print("\n--- 1. Kiểm Tra Thuộc Tính Các Nút (Nodes) ---")
    
    # User: [role_index, department_index, clearance_level, behavioral_baseline_hour] -> 4 cột
    if 'user' in data.node_types:
        user_dim = data['user'].x.shape[1]
        print(f" * Nút [user]: Số lượng = {data['user'].x.shape[0]}, Số đặc trưng = {user_dim}")
        if user_dim == 4:
            print("   -> [ĐÚNG] Đã lấy đủ 4 thuộc tính (gồm cả behavioral_baseline_hour).")
        else:
            print(f"   -> [SAI] Số đặc trưng user đang là {user_dim}, kỳ vọng phải là 4.")
            errors += 1
    else:
        print(" -> [LỖI] Không tìm thấy nút 'user' trong đồ thị!"); errors += 1

    # Session: [time_window_norm, is_internal_ip] -> 2 cột
    if 'session' in data.node_types:
        sess_dim = data['session'].x.shape[1]
        print(f" * Nút [session]: Số lượng = {data['session'].x.shape[0]}, Số đặc trưng = {sess_dim}")
        if sess_dim == 2:
            print("   -> [ĐÚNG] Đã lấy đủ 2 thuộc tính thực tế (time_window_norm, is_internal_ip).")
        else:
            print(f"   -> [SAI] Số đặc trưng session đang là {sess_dim}, kỳ vọng là 2.")
            errors += 1
    else:
        print(" -> [LỖI] Không tìm thấy nút 'session' trong đồ thị!"); errors += 1

    # Table: [row_count, security_level] -> 2 cột
    if 'table' in data.node_types:
        tbl_dim = data['table'].x.shape[1]
        print(f" * Nút [table]: Số lượng = {data['table'].x.shape[0]}, Số đặc trưng = {tbl_dim}")
        if tbl_dim == 2:
            print("   -> [ĐÚNG] Đã lấy đủ 2 thuộc tính cấu trúc bảng.")
        else:
            print(f"   -> [SAI] Số đặc trưng table đang là {tbl_dim}, kỳ vọng là 2.")
            errors += 1

    # SQL Template: [cmd_type, join_count, where_count, has_subquery] -> 4 cột
    if 'sql_template' in data.node_types:
        sql_dim = data['sql_template'].x.shape[1]
        print(f" * Nút [sql_template]: Số lượng = {data['sql_template'].x.shape[0]}, Số đặc trưng = {sql_dim}")
        if sql_dim == 4:
            print("   -> [ĐÚNG] Đã lấy đủ 4 đặc trưng định danh SQL.")
        else:
            print(f"   -> [SAI] Số đặc trưng sql_template đang là {sql_dim}, kỳ vọng là 4.")
            errors += 1

    # ================================================================
    # KHẢO SÁT CẠNH VÀ SỰ ĐỒNG BỘ MA TRẬN (EDGE FEATURES CHECK)
    # ================================================================
    print("\n--- 2. Kiểm Tra Cấu Trúc Và Thuộc Tính Cạnh (Edges) ---")

    def verify_edge(edge_tuple, expected_dim):
        nonlocal errors
        if edge_tuple in data.edge_types:
            edge_index = data[edge_tuple].edge_index
            edge_attr = data[edge_tuple].edge_attr
            
            num_edges = edge_index.shape[1]
            num_attr_rows = edge_attr.shape[0]
            attr_dim = edge_attr.shape[1]
            
            print(f" * Quan hệ {edge_tuple}:")
            print(f"   - Số lượng liên kết chỉ mục (edge_index) = {num_edges}")
            print(f"   - Số lượng hàng đặc trưng (edge_attr)   = {num_attr_rows}")
            print(f"   - Số chiều đặc trưng cạnh (edge_dim)    = {attr_dim}")

            # Kiểm tra lỗi lệch ma trận (Lỗi nghiêm trọng gây crash khi train PyG)
            if num_edges != num_attr_rows:
                print(f"   ==> [LỖI NGHIÊM TRỌNG] Lệch ma trận cạnh! Số lượng index ({num_edges}) "
                      f"khác số lượng hàng thuộc tính ({num_attr_rows})!")
                errors += 1
            else:
                print("   -> [ĐÚNG] Tính toàn vẹn cấu trúc ma trận cạnh OK (Kích thước trùng khớp).")

            # Kiểm tra số lượng cột đặc trưng
            if attr_dim == expected_dim:
                print(f"   -> [ĐÚNG] Số chiều đặc trưng cạnh khớp hoàn hảo với a01 (Kỳ vọng: {expected_dim}).")
            else:
                print(f"   ==> [SAI] Số chiều đặc trưng cạnh đang là {attr_dim}, kì vọng phải là {expected_dim}!")
                errors += 1
        else:
            print(f" * [CẢNH BÁO/LỖI] Cạnh {edge_tuple} không tồn tại trên đồ thị sinh ra!")
            errors += 1
        print("-" * 40)

    # Đánh giá các cạnh động bóc từ audit log
    # user -> opens -> session: [is_after_hours, session_duration_norm] -> 2 chiều
    verify_edge(('user', 'opens', 'session'), expected_dim=2)
    
    # user -> executes -> sql_template: [execution_count_norm] -> 1 chiều
    verify_edge(('user', 'executes', 'sql_template'), expected_dim=1)
    
    # sql_template -> belongs_to -> session: [sequence_order_norm] -> 1 chiều
    verify_edge(('sql_template', 'belongs_to', 'session'), expected_dim=1)
    
    # sql_template -> affects -> table: [rows_affected_log, permission_denied, success, execution_error] -> 4 chiều
    verify_edge(('sql_template', 'affects', 'table'), expected_dim=4)

    # ================================================================
    # KHẢO SÁT NHÃN VÀ MẶT NẠ PHÂN CHIA (LABELS & MASKS CHECK)
    # ================================================================
    print("\n--- 3. Kiểm Tra Nhãn Dự Đoán & Mask Phân Chia ---")
    if 'session' in data.node_types:
        labels = data['session'].y
        n_sessions = data['session'].x.shape[0]
        
        if labels.shape[0] == n_sessions:
            print(f" * [ĐÚNG] Số lượng nhãn ({labels.shape[0]}) khớp chính xác với số lượng nút Session.")
        else:
            print(f" * [SAI] Số lượng nhãn ({labels.shape[0]}) bị lệch khỏi số nút Session ({n_sessions})!"); errors += 1
            
        # Kiểm tra Mask Train/Val/Test
        masks = ['train_mask', 'val_mask', 'test_mask']
        all_masks_ok = True
        for m in masks:
            if hasattr(data['session'], m):
                mask_tensor = getattr(data['session'], m)
                if mask_tensor.shape[0] != n_sessions:
                    print(f"   - [SAI] Kích thước {m} không tương thích với số lượng nút Session!"); all_masks_ok = False
            else:
                print(f"   - [LỖI] Nút Session bị thiếu thuộc tính phân vùng {m}!"); all_masks_ok = False
        
        if all_masks_ok:
            print(" * [ĐÚNG] Toàn bộ hệ thống mặt nạ Train/Val/Test Mask phân bổ hợp lệ.")
            # Kiểm tra tính rò rỉ dữ liệu (Mỗi nút chỉ được nằm trong 1 tập)
            t_m = data['session'].train_mask.int()
            v_m = data['session'].val_mask.int()
            te_m = data['session'].test_mask.int()
            overlap = (t_m + v_m + te_m).max().item()
            if overlap > 1:
                print("   ==> [CẢNH BÁO RÒ RỈ] Một nút đang bị phân bổ trùng lặp vào nhiều tập cùng lúc!"); errors += 1
            else:
                print("   -> [ĐÚNG] Không có hiện tượng rò rỉ chéo dữ liệu giữa các tập.")

    # ================================================================
    # TỔNG KẾT KẾT QUẢ KIỂM TRA
    # ================================================================
    print("\n====================================================")
    if errors == 0:
        print("🎉 CHÚC MỪNG: FILE a02 ĐÃ ĐỒNG BỘ KHỚP 100% VỚI a01!")
        print("Đồ thị HeteroData đã sẵn sàng và an toàn để đưa vào huấn luyện GNN.")
    else:
        print(f"❌ THẤT BẠI: Phát hiện thấy {errors} điểm lỗi cấu trúc hoặc thiếu thuộc tính.")
        print("Vui lòng rà soát lại các vị trí sửa lỗi đã được chỉ định ở bước trước.")
    print("====================================================")

if __name__ == "__main__":
    run_graph_verification()

▶ BẮT ĐẦU KIỂM TRA ĐỒ THỊ HETERODATA (a02 VS a01)...

-> Đang kết nối đồ thị tĩnh bảng: [source_table] → [target_table]
-> Đã nạp 10 cạnh table → relates_to → table.

--- Thống kê đồ thị ---
HeteroData(
  user={ x=[30, 4] },
  table={ x=[10, 2] },
  sql_template={ x=[4, 4] },
  session={
    x=[93, 2],
    y=[93],
    train_mask=[93],
    val_mask=[93],
    test_mask=[93],
  },
  (user, opens, session)={
    edge_index=[2, 180],
    edge_attr=[180, 2],
  },
  (user, executes, sql_template)={
    edge_index=[2, 90],
    edge_attr=[90, 1],
  },
  (sql_template, belongs_to, session)={
    edge_index=[2, 1307],
    edge_attr=[1307, 1],
  },
  (sql_template, affects, table)={
    edge_index=[2, 1307],
    edge_attr=[1307, 4],
  },
  (table, relates_to, table)={ edge_index=[2, 10] }
)

Phân phối nhãn Session: Normal=82 | Anomaly=11
Train/Val/Test: 55/18/20 sessions
=== BƯỚC 2: Xây dựng đồ thị HeteroData thành công! ===
Đã lưu graph vào 'hetero_graph.pt'

[OK] Khởi tạo cấu trúc HeteroData thà